In [6]:
import tkinter as tk
from tkinter import ttk, messagebox
from pathlib import Path
import joblib


# ==========================================================
# GUI for Coastal Morphology Change Prediction
# Inputs:
# X1 = Δt  : time interval
# X2 = Tr  : tidal range
# X3 = Hs  : significant wave height
# X4 = x   : position along each coastal profile
#
# Output:
# Y = Δz   : morphology change
# ==========================================================


# -----------------------------
# Model files on Desktop
# -----------------------------
DESKTOP_DIR = Path.home() / "Desktop"

MODEL_FILES = {
    "CatBoost Gradient Boosting (CGB)": "CGB1.joblib",
    "Decision Tree (DT)": "DT1.joblib",
    "LightGBM (LGB)": "LGB1.joblib",
    "Stochastic Gradient Boosting (SGB)": "SGB1.joblib",
    "Random Forest (RF)": "RF1.joblib",
}

loaded_models = {}


# -----------------------------
# Load selected model
# -----------------------------
def load_model(model_name):
    if model_name in loaded_models:
        return loaded_models[model_name]

    model_path = DESKTOP_DIR / MODEL_FILES[model_name]

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model file not found:\n{model_path}\n\n"
            "Please make sure the selected .joblib file is on the Desktop."
        )

    model = joblib.load(model_path)
    loaded_models[model_name] = model
    return model


# -----------------------------
# Prediction function
# -----------------------------
def predict():
    try:
        selected_model = model_var.get()

        if selected_model not in MODEL_FILES:
            messagebox.showerror("Model Error", "Please select a valid model.")
            return

        model = load_model(selected_model)

        X1 = float(entry_dt.get())
        X2 = float(entry_tr.get())
        X3 = float(entry_hs.get())
        X4 = float(entry_x.get())

        input_data = [[X1, X2, X3, X4]]
        prediction = model.predict(input_data)[0]

        output_label.config(text=f"{float(prediction):.4f}")
        model_used_label.config(text=f"Model used: {selected_model}")

    except ValueError:
        messagebox.showerror(
            "Input Error",
            "Please enter valid numerical values for all inputs."
        )

    except FileNotFoundError as e:
        messagebox.showerror("Model File Error", str(e))

    except Exception as e:
        messagebox.showerror(
            "Prediction Error",
            f"An error occurred during prediction:\n\n{e}"
        )


# -----------------------------
# Clear function
# -----------------------------
def clear_inputs():
    entry_dt.delete(0, tk.END)
    entry_tr.delete(0, tk.END)
    entry_hs.delete(0, tk.END)
    entry_x.delete(0, tk.END)

    output_label.config(text="--")
    model_used_label.config(text="Model used: --")

    entry_dt.focus_set()


# -----------------------------
# Check missing model files
# -----------------------------
def check_model_files():
    missing = []

    for file_name in MODEL_FILES.values():
        model_path = DESKTOP_DIR / file_name
        if not model_path.exists():
            missing.append(file_name)

    if missing:
        missing_text = "\n".join(missing)
        messagebox.showwarning(
            "Missing Model Files",
            "The following model files were not found on the Desktop:\n\n"
            f"{missing_text}\n\n"
            "Prediction will work only for the available models."
        )


# ==========================================================
# GUI Design
# ==========================================================

root = tk.Tk()
root.title("ML-Based Coastal Morphology Change Prediction")
root.geometry("760x660")
root.resizable(False, False)
root.configure(bg="#f4f6f8")


# -----------------------------
# Styles
# -----------------------------
style = ttk.Style()
style.theme_use("clam")

style.configure(
    "Title.TLabel",
    font=("Arial", 16, "bold"),
    background="#f4f6f8",
    foreground="#1f2937"
)

style.configure(
    "Subtitle.TLabel",
    font=("Arial", 10),
    background="#f4f6f8",
    foreground="#4b5563"
)

style.configure(
    "Card.TLabelframe",
    background="#ffffff"
)

style.configure(
    "Card.TLabelframe.Label",
    font=("Arial", 11, "bold"),
    foreground="#111827",
    background="#ffffff"
)

style.configure(
    "Input.TLabel",
    font=("Arial", 11, "bold"),
    background="#ffffff",
    foreground="#111827"
)

style.configure(
    "Hint.TLabel",
    font=("Arial", 9),
    background="#ffffff",
    foreground="#6b7280"
)

style.configure(
    "Result.TLabel",
    font=("Arial", 22, "bold"),
    background="#ffffff",
    foreground="#b91c1c"
)

style.configure(
    "TEntry",
    font=("Arial", 12),
    padding=5
)

style.configure(
    "TButton",
    font=("Arial", 11, "bold"),
    padding=8
)

style.configure(
    "TCombobox",
    font=("Arial", 11),
    padding=5
)


# -----------------------------
# Header
# -----------------------------
header_frame = tk.Frame(root, bg="#f4f6f8")
header_frame.pack(fill="x", padx=20, pady=(20, 10))

title_label = ttk.Label(
    header_frame,
    text="Machine Learning-Based Prediction of Coastal Morphology Change",
    style="Title.TLabel"
)
title_label.pack(anchor="center")

subtitle_label = ttk.Label(
    header_frame,
    text="Inputs: Δt, Tr, Hs, and x     |     Output: morphology change, Δz",
    style="Subtitle.TLabel"
)
subtitle_label.pack(anchor="center", pady=(5, 0))


# -----------------------------
# Model selection
# -----------------------------
model_frame = ttk.LabelFrame(
    root,
    text="Model Selection",
    style="Card.TLabelframe",
    padding=(15, 12)
)
model_frame.pack(fill="x", padx=20, pady=8)

ttk.Label(
    model_frame,
    text="Select ML model:",
    style="Input.TLabel"
).grid(row=0, column=0, padx=5, pady=8, sticky="w")

model_var = tk.StringVar()
model_var.set("CatBoost Gradient Boosting (CGB)")

model_dropdown = ttk.Combobox(
    model_frame,
    textvariable=model_var,
    values=list(MODEL_FILES.keys()),
    state="readonly",
    width=45
)
model_dropdown.grid(row=0, column=1, padx=10, pady=8, sticky="w")


# -----------------------------
# Input parameters
# -----------------------------
input_frame = ttk.LabelFrame(
    root,
    text="Input Parameters",
    style="Card.TLabelframe",
    padding=(15, 12)
)
input_frame.pack(fill="x", padx=20, pady=8)

ttk.Label(input_frame, text="Time interval", style="Input.TLabel").grid(row=0, column=0, padx=5, pady=8, sticky="w")
ttk.Label(input_frame, text="X1 = Δt", style="Hint.TLabel").grid(row=0, column=1, padx=5, pady=8, sticky="w")
entry_dt = ttk.Entry(input_frame, width=25)
entry_dt.grid(row=0, column=2, padx=10, pady=8)

ttk.Label(input_frame, text="Tidal range", style="Input.TLabel").grid(row=1, column=0, padx=5, pady=8, sticky="w")
ttk.Label(input_frame, text="X2 = Tr", style="Hint.TLabel").grid(row=1, column=1, padx=5, pady=8, sticky="w")
entry_tr = ttk.Entry(input_frame, width=25)
entry_tr.grid(row=1, column=2, padx=10, pady=8)

ttk.Label(input_frame, text="Significant wave height", style="Input.TLabel").grid(row=2, column=0, padx=5, pady=8, sticky="w")
ttk.Label(input_frame, text="X3 = Hs", style="Hint.TLabel").grid(row=2, column=1, padx=5, pady=8, sticky="w")
entry_hs = ttk.Entry(input_frame, width=25)
entry_hs.grid(row=2, column=2, padx=10, pady=8)

ttk.Label(input_frame, text="Position along coastal profile", style="Input.TLabel").grid(row=3, column=0, padx=5, pady=8, sticky="w")
ttk.Label(input_frame, text="X4 = x", style="Hint.TLabel").grid(row=3, column=1, padx=5, pady=8, sticky="w")
entry_x = ttk.Entry(input_frame, width=25)
entry_x.grid(row=3, column=2, padx=10, pady=8)


# -----------------------------
# Prediction result
# -----------------------------
result_frame = ttk.LabelFrame(
    root,
    text="Prediction Result",
    style="Card.TLabelframe",
    padding=(15, 12)
)
result_frame.pack(fill="x", padx=20, pady=8)

ttk.Label(
    result_frame,
    text="Predicted morphology change (m):",
    style="Input.TLabel"
).grid(row=0, column=0, padx=5, pady=8, sticky="w")

ttk.Label(
    result_frame,
    text="Δz =",
    style="Input.TLabel"
).grid(row=0, column=1, padx=5, pady=8, sticky="w")

output_label = ttk.Label(
    result_frame,
    text="--",
    style="Result.TLabel"
)
output_label.grid(row=0, column=2, padx=8, pady=8, sticky="w")

model_used_label = ttk.Label(
    result_frame,
    text="Model used: --",
    style="Hint.TLabel"
)
model_used_label.grid(row=1, column=0, columnspan=3, padx=5, pady=3, sticky="w")


# -----------------------------
# Prediction button
# -----------------------------
button_frame = tk.Frame(root, bg="#f4f6f8")
button_frame.pack(pady=15)

predict_button = ttk.Button(
    button_frame,
    text="Predict Morphology Change (Δz)",
    command=predict
)
predict_button.grid(row=0, column=0, padx=8)

clear_button = ttk.Button(
    button_frame,
    text="Clear",
    command=clear_inputs
)
clear_button.grid(row=0, column=1, padx=8)

exit_button = ttk.Button(
    button_frame,
    text="Exit",
    command=root.destroy
)
exit_button.grid(row=0, column=2, padx=8)


# -----------------------------
# Start GUI
# -----------------------------
entry_dt.focus_set()
check_model_files()
root.mainloop()

d:\Anaconda\envs\new_ml_env\lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(
